<img src="https://drive.google.com/uc?export=view&id=19S-shKxcBeLwB9sA_VYNiWTzNwMtpavY" width="100%" align="middle"/>


# <font color="#FF0000"><center> **NLOS: Viendo más allá de las paredes 🐇🔦** </center></font>

## <font color='#00BFFF'> **Contenido** <a name="tema1">

[<font color='#00BFFF'>**1. Non-line-of-sight**</font>](#tema1)

[<font color='#00BFFF'>**2. Simulación NLOS**</font>](#tema2)

[<font color='#00BFFF'>**3. Configuración de la escena**</font>](#tema4)

[<font color='#00BFFF'>**4. Visualización de los datos adquiridos**</font>](#tema5)

[<font color='#00BFFF'>**5. Reconstrucción**</font>](#tema6)

[<font color='#00BFFF'>**6. Referencias**</font>](#referencias)

# <font color="#00BFFF"> <center> **Non-line-of-sight** </center> </font> <a name="tema1">

<font color="#FF0000">**Non-Line-of-Sight imaging**</font> es una técnica que permite reconstruir objetos ocultos que no están directamente visibles para una cámara o sensor. Esto se logra analizando la luz que se refleja indirectamente en superficies visibles, como paredes o pisos. Por ejemplo, al iluminar una pared cercana con un láser, parte de esa luz puede rebotar hacia un objeto oculto y luego regresar a la pared. Capturando y analizando estas reflexiones, es posible inferir información sobre el objeto escondido, como su forma o posición.

<div style="display: flex; justify-content: space-around;">
  <img src="https://drive.google.com/uc?export=view&id=15bDhpEjpxtimWGOTxdZxkx8Oj3u6CRoK" width="33%">
  <img src="https://drive.google.com/uc?export=view&id=1e6Ap4VlMLn6dMy39QtlLsljGIA7_H_k4" width="33%">
  <img src="https://drive.google.com/uc?export=view&id=1cP0hE7SLyahTtk9VBeYJijpNhtD6eOiK" width="33%">
</div>



# <font color="#00BFFF"> <center> **Simulación NLOS** </center> </font> <a name="tema2">

El propósito de este notebook es <font color="#FF0000">**simular el proceso detrás de la reconstrucción de escenas en condiciones de Non-Line-of-Sight (NLOS)**</font>, es decir, cuando el objeto de interés no se encuentra visible al sensor. Para lograr dicha reconstrucción, partiremos de una configuración específica que define la disposición del sistema de captura y los elementos de la escena:
<img src="https://drive.google.com/uc?export=view&id=1y-5aDsZ9pPQw4pppGL61m4Au6tLEnAVh" width="70%">

Bajo este planteamiento, el proceso completo será desarrollado siguiendo una secuencia de etapas, descritas en el siguiente esquema:

<img src="https://drive.google.com/uc?export=view&id=1DOnK-kCK-igZe5y-zN8A3JXxi6XQpfbn" width="100%">

##Librerias de simulación (Mitsuba 3, mitransient, y-tal)
<div style="display: flex; justify-content: space-around;">
  <img src="https://drive.google.com/uc?export=view&id=1WpR-TLMX0dl-2JMLqMf2hJXloF8ZgIDh" width="33%">
  <img src="https://drive.google.com/uc?export=view&id=15ZYN82mH1hGrWesvMsM2TFU3H53esePB" width="33%">
  <img src="https://drive.google.com/uc?export=view&id=1DERKFbQ8Ch1xfpKd9mBAG2n1D5YrZYyM" width="33%">
</div>


In [ ]:
#@title Instalación de librerías
!pip install -q mitransient==1.1.1
!pip install -q mitsuba==3.6.0
!pip install -q drjit==1.0.1
!pip install trimesh plotly

In [ ]:
#@title Cargamos librerías
import numpy as np
import drjit as dr
import mitsuba as mi
mi.set_variant('cuda_ad_rgb')
import mitransient as mitr
import matplotlib.pyplot as plt
from mitsuba import ScalarTransform4f as T
import trimesh

In [ ]:
#@title Funciones necesarias

def load_from_drive(file_id, name):
  gdown.download(id=file_id, output=name, quiet=False)
  return name

def normalize_obj(input_file, output_file, target_size=1.0):
  mesh = trimesh.load(input_file)

  bbox = mesh.bounding_box_oriented
  center = bbox.centroid
  scale = bbox.extents.max()

  mesh.apply_translation(-center)
  mesh.apply_scale(target_size / scale)

  mesh.export(output_file)

def rotate_obj(input_file, output_file, rotation_axis, rotation_angle):
  mesh = trimesh.load(input_file)

  rotation_axis = rotation_axis / np.linalg.norm(rotation_axis)

  rotation_matrix = trimesh.transformations.rotation_matrix(
      angle=np.deg2rad(rotation_angle),
      direction=rotation_axis,
      point=[0, 0, 0]
  )

  mesh.apply_transform(rotation_matrix)

  mesh.export(output_file)

def plot_obj(input_file):
  mesh = trimesh.load(input_file)

  fig = go.Figure(data=[
    go.Mesh3d(
        x=mesh.vertices[:, 0],
        y=mesh.vertices[:, 1],
        z=mesh.vertices[:, 2],
        i=mesh.faces[:, 0],
        j=mesh.faces[:, 1],
        k=mesh.faces[:, 2],
        color='green',
        opacity=0.6
    )
  ])
  fig.update_layout(scene_aspectmode='data')
  fig.show()


# <font color="#00BFFF"> <center> **Configuración de la escena** </center> </font> <a name="tema4">

In [ ]:
#@title Funciones de previsualización y creación de la habitación

def create_room(T, width, height, depth, wall_thickness, floor_width, floor_depth, door_width, fov, angle_fov, flip_norms=True):
    '''
    eje x : width
    eje y : height
    eje z : depth
    '''

    if flip_norms:
      normals = [True, False, True, False, True]
    else:
      normals = [False, True, False, True, False]

    if width < door_width:
      print('The width of the door must be greater than the width of the room')

    left_wall = mi.load_dict({
        'type': 'cube',
        'flip_normals': normals[0],
        'to_world': T()
            .translate([-width + (wall_thickness), 0, 0])
            .scale([wall_thickness, height, depth - wall_thickness]),
        'bsdf': {
            'type': 'diffuse',
            'reflectance': {
                'type': 'rgb',
                'value': [1, 1, 1]
            }
        }
    })

    right_wall = mi.load_dict({
        'type': 'cube',
        'flip_normals': normals[1],
        'to_world': T()
            .translate([width - (wall_thickness), 0, 0])
            .scale([wall_thickness, height, depth - wall_thickness]),
        'bsdf': {
            'type': 'diffuse',
            'reflectance': {
                'type': 'rgb',
                'value': [1, 1, 1]
            }
        }
    })

    back_wall = mi.load_dict({
        'type': 'cube',
        'flip_normals': normals[2],
        'to_world': T()
            .translate([0, 0, -depth])
            .scale([width, height, wall_thickness]),
        'bsdf': {
            'type': 'diffuse',
            'reflectance': {
                'type': 'rgb',
                'value': [1, 1, 1]
            }
        }

    })

    front_wall = mi.load_dict({
        'type': 'cube',
        'flip_normals': normals[3],
        'to_world': T()
            .translate([-door_width, 0, depth])
            .scale([width-door_width, height, wall_thickness]),
        'bsdf': {
            'type': 'diffuse',
            'reflectance': {
                'type': 'rgb',
                'value': [1, 1, 1]
            }
        }
    })

    floor = mi.load_dict({
      'type': 'cube',
      'flip_normals': normals[4],
      'to_world': T()
      .translate([0, -(height+(wall_thickness/2)), 0])
      .scale([floor_width, wall_thickness, floor_depth]),
      'bsdf': {
          'type': 'diffuse',
          'reflectance': {
              'type': 'rgb',
              'value': [1, 1, 1]
          }
      }
      })

    theta = np.deg2rad(angle_fov + 90)
    y_offset = (fov / 2) * np.abs(np.sin(theta))
    z_offset = (fov / 2) * (1 - np.cos(theta))

    sensor_fov = mi.load_dict({
        'type': 'rectangle',
        'to_world': T()
            .translate([
                width - 2 * door_width,
                (wall_thickness/2) - height + 0.001 + y_offset,
                depth + wall_thickness + (fov/2) - z_offset
            ])
            .rotate([1, 0, 0], angle_fov)
            .scale([fov/2, fov/2, wall_thickness]),
        'bsdf': {
            'type': 'diffuse',
            'reflectance': {
                'type': 'rgb',
                'value': [1, 0, 0]
            }
        }
    })

    return right_wall, left_wall, back_wall, front_wall, floor, sensor_fov

def preview_room(right_wall, left_wall, back_wall, front_wall, floor, sensor_fov, emitters):
  rgb_integrator = mi.load_dict({'type': 'path'})
  norm_integrator = mi.load_dict({'type': 'aov', 'aovs': 'nn:sh_normal'})

  sensor = mi.load_dict(
      {
          'type': 'perspective',
          'to_world': mi.ScalarTransform4f().look_at(
              origin=[1.5, 10, 3],
              target=[1.5, 0, 0],
              up=[0, 1, 0]
          ),
          'fov': 60,
          'film': {
              'type': 'hdrfilm',
              'width': 256,
              'height': 256,
              'rfilter': {"type": "box"}
          },
          'sampler': {
              'type': 'independent',
              'sample_count': 64
          }
      }
  )

  rgb_scene = mi.load_dict(
      {
          'type': 'scene',
          'integrator': rgb_integrator,
          'sensor': sensor,
          'rightwall': right_wall,
          'leftwall': left_wall,
          'backwall': back_wall,
          'frontwall': front_wall,
          'floor': floor,
          'sensor_fov': sensor_fov,
          **emitters
      }
  )

  norm_scene = mi.load_dict(
      {
          'type': 'scene',
          'integrator': norm_integrator,
          'sensor': sensor,
          'rightwall': right_wall,
          'leftwall': left_wall,
          'backwall': back_wall,
          'frontwall': front_wall,
          'floor': floor,
          'sensor_fov': sensor_fov,
          **emitters
      }
  )

  rgb_img = mi.render(rgb_scene)
  norm_img = mi.render(norm_scene)
  plt.subplot(1, 2, 1)
  plt.imshow(np.clip(rgb_img, 0, 1))
  plt.axis('off')
  plt.subplot(1, 2, 2)
  plt.imshow(np.clip(norm_img, 0, 1))
  plt.axis('off')
  plt.show()

def plot_transient_info(data_transient, data_steady, i, j, t):
  plt.figure(figsize=(15, 10))
  plt.subplot(2, 2, 1)
  plt.plot(np.array(data_transient)[i, j, :, 0])
  plt.xlabel('Time index')
  plt.ylabel(f'Captured radiance at pixel ({i}, {j})')
  plt.subplot(2, 2, 2)
  plt.imshow(np.flip(np.array(data_transient)[:, :, t, 0], axis=0), cmap='hot')
  plt.colorbar()
  plt.axis('off')
  plt.xlabel('Relay wall X')
  plt.ylabel('Relay wall Y')
  plt.title(f't_idx = {t}')
  plt.subplot(2, 2, 3)
  plt.plot(np.sum(np.array(data_transient)[:, :, :, 0], axis=(0, 1)))
  plt.xlabel('Time index')
  plt.ylabel('Total captured radiance (all pixels)')
  plt.subplot(2, 2, 4)
  plt.imshow(np.clip(np.flip(data_steady, axis=0) * 10, 0, 1)) ##SCALING!!!
  plt.show()
  curve = np.sum(np.array(data_transient)[:, :, :, 0], axis=(0,1))

def nlos_transient_render(x, y, z, right_wall, left_wall, back_wall, front_wall, floor, sensor_fov, emitter):
  transient_film = mi.load_dict({
      'type': 'transient_hdr_film',
      'width': 32,
      'height': 32,
      'temporal_bins': 900,
      'bin_width_opl': 0.009,
      'start_opl': 1,
      'rfilter': {'type': 'box'}
  })

  nlos_sensor = mi.load_dict({
      'type': 'nlos_capture_meter',
      'sampler': {'type': 'independent',
                  'sample_count': 25000},
      'account_first_and_last_bounces': True,
      'sensor_origin': [x, y + 0.25, z],
      'transient_film': transient_film
  })

  geometry = mi.load_dict({
    'type': 'cube',
    'to_world': T()
    .translate(mi.ScalarPoint3f(x + x_pos, y + y_pos, z + z_pos))
    .scale([0.5, 1, 0.01]),
    'bsdf': {
        'type': 'diffuse',
        'reflectance': {
            'type': 'rgb',
            'value': [1, 1, 1]
        }
    }
  })

  params = mi.traverse(sensor_fov)
  M = np.array(params['to_world'].matrix).squeeze()

  tx = M[0, 3]
  ty = M[1, 3]
  tz = M[2, 3]

  sx = np.linalg.norm(M[:3, 0])
  sy = np.linalg.norm(M[:3, 1])
  sz = np.linalg.norm(M[:3, 2])

  R = np.zeros((3, 3))
  R[:, 0] = M[:3, 0] / sx
  R[:, 1] = M[:3, 1] / sy
  R[:, 2] = M[:3, 2] / sz

  trace = np.trace(R)
  angle = np.arccos(np.clip((trace - 1) / 2, -1, 1))

  if abs(angle) < 1e-8:
    axis = np.array([1.0, 0.0, 0.0])
  else:
    axis = np.array([
        R[2,1] - R[1,2],
        R[0,2] - R[2,0],
        R[1,0] - R[0,1]
        ])
    axis = axis / (np.linalg.norm(axis) + 1e-12)

  angle_fov = float(angle * 180 / np.pi)
  axis = axis.tolist()

  relay = mi.load_dict({
      'type': 'rectangle',
      'to_world': T()
        .translate([tx, ty, tz])
        .rotate(axis, angle_fov)
        .scale([sx, sy, sz]),
      'bsdf': {'type': 'diffuse',
               'reflectance': 1.0},
      'nlos_sensor': nlos_sensor
  })

  integrator = mi.load_dict({
      'type': 'transient_nlos_path',
      'nlos_laser_sampling': False,
      'nlos_hidden_geometry_sampling': False,
      'nlos_hidden_geometry_sampling_do_rroulette': False,
      "temporal_filter": 'box'
  })

  scene = mi.load_dict({
      'type': 'scene',
      'integrator': integrator,
      'sensor': relay,
      'rightwall': right_wall,
      'leftwall': left_wall,
      'backwall': back_wall,
      'frontwall': front_wall,
      'floor': floor,
      'emitter': emitter,
      'geometry': geometry
  })


  transient_integrator = scene.integrator()
  data_steady, data_transient = transient_integrator.render(scene,spp=500000)
  return data_transient, data_steady

In [ ]:
#@title Previsualización de la habitación

room_width = 3 # @param {type:"number"}
room_height = 2 # @param {type:"number"}
room_depth = 3 # @param {type:"number"}
wall_thickness = 0.1 # @param {type:"number"}
floor_width = 20 # @param {type:"number"}
floor_depth = 20 # @param {type:"number"}
door_width = 1.5 # @param {type:"number"}
fov_sensor = 1 # @param {type:"number"}
angle_fov = -90 # @param {type:"number"}
flip_normals = False # @param {type:"boolean"}
directional_emitter = True # @param {type:"boolean"}
sphere_emitter = False # @param {type:"boolean"}
facet_emitter = True # @param {type:"boolean"}
projector_emitter = True # @param {type:"boolean"}
z_object = False # @param {type:"boolean"}

r, l, b, f, g, s = create_room(
    T,
    room_width,
    room_height,
    room_depth,
    wall_thickness,
    floor_width,
    floor_depth,
    door_width,
    fov_sensor,
    angle_fov,
    flip_normals
)

params = mi.traverse(s)
transform = params['to_world'].matrix
x = transform[0, 3][0]
y = transform[1, 3][0]
z = transform[2, 3][0]

emitters = {}

if directional_emitter:
    x_dir = -1 # @param {type:"number"}
    y_dir = -1 # @param {type:"number"}
    z_dir = -1 # @param {type:"number"}
    emitters['dir_light'] = {
        'type': 'directional',
        'direction': [x_dir, y_dir, z_dir],
        'irradiance': {
            'type': 'rgb',
            'value': [2, 2, 2]
        }
    }

if facet_emitter:
    x_pos = 0 # @param {type:"number"}
    y_pos = 1 # @param {type:"number"}
    z_pos = -2 # @param {type:"number"}
    emitters['facet_emitter'] = {
    'type': 'cube',
    'to_world': T()
    .translate(mi.ScalarPoint3f(x + x_pos, y + y_pos, z + z_pos))
    .scale([0.5, 1, 0.01]),
    'bsdf': {
        'type': 'diffuse',
        'reflectance': {
            'type': 'rgb',
            'value': [1, 1, 1]
        }
    }
    }

if sphere_emitter:
    x_pos = 0 # @param {type:"number"}
    y_pos = 1 # @param {type:"number"}
    z_pos = -2.5 # @param {type:"number"}
    radius = 0.2 # @param {type:"number"}
    emitters['sphere_emitter'] = {
    'type': 'sphere',
    'to_world': T()
    .translate(mi.ScalarPoint3f(x + x_pos, y + y_pos, z + z_pos)),
    'radius': radius,
    'emitter': {
        'type': 'area',
        'radiance': {
            'type': 'rgb',
            'value': [1, 1, 1]
            }
        }
    }
if projector_emitter:
    x_aim = 0.2 #@param {type:"number"}
    y_aim = 0 #@param {type:"number"}
    z_aim = -0.65 #@param {type:"number"} empezó -0.65
    fov_emitter = 10 #@param {type:"number"}
    emitters['projector_emitter'] = {
        'type': 'projector',
        'irradiance': 1000.0,
        'fov': fov_emitter,
        'to_world': T().look_at(
            origin=[0, 0, 3],
            target=[x + x_aim, y + y_aim, z + z_aim],
            up=[1, 0, 0]
        )
    }

preview_room(r, l, b, f, g, s, emitters)

In [ ]:
#@title Resultados renderizado transitorio
data_transient, data_steady= nlos_transient_render(x, y, z, r, l, b, f, g, s, emitters['projector_emitter'])

In [ ]:
mitr.visualization.show_video(np.moveaxis(data_transient*10000, 0, 1), 2)

In [ ]:
plot_transient_info(data_transient, data_steady, 0, 0, 100)

<div style="background-color:#a82757; padding:14px; border-left:6px solid #f2facf; border-radius:6px">

## **RETO 1 — Cambiar el tamaño de la habitación**


Modifica:

- `room_width`
- `room_height`
- `room_depth`

y observa cómo cambia la visualización transient.

Preguntas guía:

1. ¿Qué ocurre con los tiempos de llegada cuando el cuarto es más grande?
2. ¿La energía llega más rápido o más lento?

</div>

<div style="background-color:#a82757; padding:14px; border-left:6px solid #f2facf; border-radius:6px">

## **RETO 2 — Explorar el campo de visión del emisor**


Cambia el valor de:

- `fov_emitter`

y analiza cómo cambia la iluminación indirecta.

Preguntas guía:

1. ¿Qué ocurre cuando el FOV es muy pequeño?
2. ¿Qué pasa cuando el FOV es demasiado grande?

</div>

# <font color="#00BFFF"> <center> **Datos y visualización** </center> </font> <a name="tema3">

Para representar los objetos ocultos en la escena, se emplearán modelos 3D definidos mediante archivos `.obj`, los cuales describen la geometría mediante mallas triangulares, facilitando su manipulación y visualización en entornos computacionales.
<div style="display: flex; justify-content: space-around;">
  <img src="https://drive.google.com/uc?export=view&id=13TgfLi5KvAZBuWFPZmxJv31n5X2CnbVl" width="36.5%">
  <img src="https://drive.google.com/uc?export=view&id=1SX_5FC8nPrtVbldsnHbw9-j2afLTbapr" width="34%">
  <img src="https://drive.google.com/uc?export=view&id=1BhF3g6Dxh1Hhoq19-JQHj19iZy11l2sy" width="23%">
</div>


In [ ]:
#@title Carga de los datos
%%capture
import gdown

load_from_drive('1UXZHN5pkTqzH_S6XbaVAQ3jPknDLbZMf', 'objs.zip')
!unzip "/content/objs.zip" -d "/content"

In [ ]:
#@title Visualización de los objetos 3D
import plotly.graph_objects as go
plot_obj('/content/objs/z.obj')

# <font color="#00BFFF"> <center> **Reconstrucción** </center> </font> <a name="tema6">

In [ ]:
#@title Configuración
config_content = """MITSUBA_VERSION=3
MITSUBA2_TRANSIENT_NLOS_FOLDER=
MITSUBA3_TRANSIENT_NLOS_FOLDER=
LOG_LEVEL=INFO"""

with open('/root/.tal.conf', 'w') as f:
    f.write(config_content)

print("Contenido de .tal.conf:")
!cat /root/.tal.conf

In [ ]:
#@title Carga de las escenas
%%capture
load_from_drive('1vzeSkNGrzYvxV3NqZnMk5LEalQlk0p_7', 'yamls.zip')
!unzip "/content/yamls.zip" -d "/content"

In [ ]:
#@title Renderizado con y-tal
!tal render /content/yamls/nlos-z.yaml

In [ ]:
#@title Lectura de la data_transient generada por y-tal
import tal, os
root = '/content/yamls/20260525-081219/nlos-z.hdf5'
data = tal.io.read_capture(os.path.join(root))

In [ ]:
#@title Reconstrucción mediante FPB
volume_xyz = tal.reconstruct.get_volume_project_rw(data, depths=[1.0,])

H_1 = tal.reconstruct.fbp.solve(data,
                                wl_mean=0.07, wl_sigma=0.029,
                                volume_xyz=volume_xyz, camera_system=tal.enums.CameraSystem.DIRECT_LIGHT)
tal.plot.amplitude_phase(H_1, title='Filtered backprojection (fbp)')

In [ ]:
#@title Ground Truth
depth = data.scene_info['ground_truth']['depth'][..., 2]
plt.imshow(depth, cmap='gray')
plt.show()

In [ ]:
#@title Comprobación de las normales
normals = data.scene_info['ground_truth']['normals']
plt.imshow(np.clip(normals * -1, 0, 1))
plt.show()


# <font color="#00BFFF"> <center> **Referencias** </center> </font> <a name="referencias">

[1.Non-Line-of-Sight Imaging Edmund Optics](https://www.edmundoptics.com/knowledge-center/trending-in-optics/non-line-of-sight-imaging/)

[2.Mitsuba 3 documentation](https://mitsuba.readthedocs.io/en/stable/index.html#)

[3.Mitransient documentation](https://mitransient.readthedocs.io/en/latest/index.html#)

[4.(Your)-Transient Auxiliary Library repository](https://github.com/diegoroyo/tal/tree/master)

[5.Multidimensional Arrays in C – 2D and 3D Arrays](https://www.geeksforgeeks.org/multidimensional-arrays-in-c/)

```javascript
@software{jakob2022mitsuba3,
    title = {Mitsuba 3 renderer},
    author = {Wenzel Jakob and Sebastien Speierer
    and Nicolas Roussel and Merlin Nimier-David
    and Delio Vicini and Tizian Zeltner
    and Baptiste Nicolet and Miguel Crespo
    and Vincent Leroy and Ziyi Zhang},
    note = {https://mitsuba-renderer.org},
    version = {3.0.1},
    year = 2022,
}
```

```javascript
@misc{mitransient,
      title        = {mitransient},
      author       = {Royo, Diego and Crespo, Miguel and Garcia-Pueyo, Jorge},
      year         = 2023,
      journal      = {GitHub repository},
      doi          = {https://doi.org/10.5281/zenodo.11032518},
      publisher    = {GitHub},
      howpublished = {\url{https://github.com/diegoroyo/mitransient}}
}
```

```javascript
@article{royo2022non,
	title        = {Non-line-of-sight transient rendering},
	author       = {Diego Royo and Jorge Garcia and Adolfo Munoz and Adrian Jarabo},
	year         = 2022,
	journal      = {Computers & Graphics},
	doi          = {https://doi.org/10.1016/j.cag.2022.07.003},
	issn         = {0097-8493},
	url          = {https://www.sciencedirect.com/science/article/pii/S0097849322001200}
}
```

```javascript
@software{Royo_y-tal,
author = {Royo, Diego and Luesia-Lahoz, Pablo},
license = {GPL-3.0},
title = {{y-tal}},
url = {https://github.com/diegoroyo/tal},
publisher = {GitHub},
doi = {https://doi.org/10.5281/zenodo.11197745},
}
```
